<a href="https://colab.research.google.com/github/winniewyl/super-bowl-ad-analysis/blob/main/notebooks/scrape_superbowl_ads_com.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Scrape Homepage Links and Detect YouTube Embeds
This notebook scrapes all homepage links from https://www.superbowl-ads.com/ and checks which of those links directly contain YouTube videos (either in `<iframe>` tags or anchor tags).

In [ ]:
# ✅ Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import time
import pandas as pd

# ---------- Setup ----------
BASE_URL = "https://www.superbowl-ads.com/"
OUTPUT_DIR = "../data/raw/youtube_gemini"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- Step 1: Scrape All Homepage Links ----------
response = requests.get(BASE_URL, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(response.text, 'html.parser')

homepage_links = []
for a_tag in soup.find_all('a', href=True):
    href = a_tag['href']
    homepage_links.append(href)

with open(f"{OUTPUT_DIR}/homepage_links.txt", "w", encoding="utf-8") as f:
    for link in homepage_links:
        f.write(link + "\n")

print(f"✅ {len(homepage_links)} total links saved to homepage_links.txt")

✅ 366 total links saved to homepage_links.txt


In [ ]:
# ---------- Step 2: Check which links contain YouTube embeds (No Duplicates) ----------
full_links = []
seen_links = set()

for link in homepage_links:
    # Construct full URL
    if link.startswith("http"):
        full_url = link
    elif link.startswith("/"):
        full_url = BASE_URL.rstrip("/") + link
    else:
        continue

    # Normalize URL
    full_url = full_url.rstrip("/").lower()

    if full_url not in seen_links:
        full_links.append(full_url)
        seen_links.add(full_url)

youtube_pages = set()

for url in full_links:
    try:
        res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        found = False

        # Check for YouTube iframes
        for iframe in soup.find_all("iframe"):
            src = iframe.get("src", "")
            if "youtube.com" in src or "youtu.be" in src:
                found = True
                break

        # Check for YouTube links
        if not found:
            for a in soup.find_all("a", href=True):
                href = a['href']
                if "youtube.com/watch" in href or "youtu.be" in href:
                    found = True
                    break

        if found:
            youtube_pages.add(url)
            print(f"✅ YouTube found: {url}")

    except Exception as e:
        print(f"⚠️ Error processing {url}: {e}")

    time.sleep(0.5)

# Save results
output_file = f"{OUTPUT_DIR}/homepage_youtube_links.txt"
with open(output_file, "w", encoding="utf-8") as f:
    for link in sorted(youtube_pages):
        f.write(link + "\n")

print(f"\n🎯 Total unique pages with YouTube links: {len(youtube_pages)}")


✅ YouTube found: https://www.superbowl-ads.com
✅ YouTube found: https://superbowl-ads.com/super-bowl-commercial-archive-video/2009-ads-video
✅ YouTube found: https://youtu.be/kdorky-13ak?si=kgmubj7movsdjm1z
✅ YouTube found: https://www.superbowl-ads.com/nike-super-bowl-lix-2025-ad-so-win
✅ YouTube found: https://www.superbowl-ads.com/salesforce-super-bowl-lix-2025-ad-gate-expectations-with-matthew-mcconaughey
✅ YouTube found: https://www.superbowl-ads.com/universal-pictures-super-bowl-lix-2025-ad-how-to-train-your-dragon
✅ YouTube found: https://www.superbowl-ads.com/google-pixel-super-bowl-lix-2025-ad-dream-job
✅ YouTube found: https://www.superbowl-ads.com/stella-artois-super-bowl-lix-2025-ad-david-dave-the-other-david
✅ YouTube found: https://www.superbowl-ads.com/yahoo-super-bowl-lix-2025-ad-email-bill-murray
✅ YouTube found: https://www.superbowl-ads.com/coors-light-super-bowl-lix-2025-ad-slow-monday
✅ YouTube found: https://www.superbowl-ads.com/little-ceasars-pizza-super-bowl-li

/tmp/ipython-input-8-2051938595.py:26: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(res.text, "html.parser")


✅ YouTube found: https://www.superbowl-ads.com/super-bowl-commercial-archive-video/2009-ads-video

🎯 Total unique pages with YouTube links: 79
